## Importando bibliotecas

In [ ]:
# Bibliotecas para manipulação de dados, visualizações, testes estatísticos e agrupamento.
# Todas importadas no início para facilitar a reprodutibilidade do notebook.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
import os

## Lendo os dados

Os dois datasets são carregados a partir de caminhos relativos ao notebook, garantindo portabilidade entre sistemas operacionais.

In [ ]:
# Construção dos caminhos com os.path.join para compatibilidade entre sistemas operacionais.
telecom_df_path = os.path.join(os.getcwd(), '..', 'data', 'telecom_dataset_new.csv')
clients_df_path = os.path.join(os.getcwd(), '..', 'data', 'telecom_clients.csv')

# Leitura dos CSVs — o pandas infere automaticamente os tipos de coluna na leitura.
telecom_df = pd.read_csv(telecom_df_path)
clients_df = pd.read_csv(clients_df_path)

# Resumo do carregamento exibido como DataFrame para facilitar a leitura.
resumo = pd.DataFrame({
    'dataset': ['telecom_df', 'clients_df'],
    'linhas': [telecom_df.shape[0], clients_df.shape[0]],
    'colunas': [telecom_df.shape[1], clients_df.shape[1]]
}).set_index('dataset')

display(resumo)

Ambos os datasets foram carregados com sucesso. O `telecom_df` tem 53.902 registros de chamadas em 9 colunas; o `clients_df` tem 732 clientes cadastrados em 3 colunas.

## Explorando os dados

Antes de calcular qualquer KPI ou testar hipóteses, é fundamental entender a estrutura, a qualidade e as distribuições dos dados. Esta seção cobre:

1. **Volume** — dimensões, período e contagem de entidades únicas
2. **Qualidade** — valores ausentes, duplicatas, tipagem e consistência lógica
3. **Resumo estatístico** — estatísticas descritivas e distribuições
4. **Variáveis mais relevantes** — mapeamento para os KPIs do projeto

---

### 2.1 Volume dos dados

Verificamos o tamanho de cada dataset, o período coberto pelos registros e a contagem de entidades únicas (operadores e clientes).

In [ ]:
# Criamos a função volume_report para não repetir o mesmo bloco de código duas vezes —
# queremos exibir as dimensões tanto do telecom_df quanto do clients_df com a mesma lógica.
# Encapsular em função torna o código reutilizável e mais fácil de manter.
def volume_report(df, name):
    rows, cols = df.shape  # .shape retorna uma tupla (nº de linhas, nº de colunas); desestruturamos nas variáveis rows e cols
    print(f'Dataset: {name}')
    print(f'  Linhas  : {rows:,}')            # :, formata o número com separador de milhar (ex.: 53.902)
    print(f'  Colunas : {cols} — {list(df.columns)}\n')  # list() converte o índice de colunas em lista legível; \n pula linha


# Chamamos a mesma função para os dois datasets — reutilização que justifica a abstração acima.
volume_report(telecom_df, 'telecom_df')
volume_report(clients_df, 'clients_df')

# Parse temporário de 'date' apenas para extrair o intervalo coberto pelos dados.
# Fazemos isso aqui (antes da etapa de limpeza) só para fins informativos.
# A conversão definitiva, com tratamento adequado do fuso horário, ocorre na limpeza.
date_parsed = pd.to_datetime(telecom_df['date'], utc=True)  # utc=True normaliza o fuso +03:00 para UTC

# Nomes das variáveis:
#   n_ops       = número (n) de operadores únicos no telecom_df
#   n_clients_t = número de clientes únicos no dataset de telecom  (t = telecom)
#   n_clients_c = número de clientes únicos no dataset de clientes (c = clients)
n_ops       = telecom_df['operator_id'].nunique()  # nunique() conta valores distintos, ignorando NaN por padrão
n_clients_t = telecom_df['user_id'].nunique()
n_clients_c = clients_df['user_id'].nunique()

print(f'Período coberto           : {date_parsed.min().date()} → {date_parsed.max().date()}')
print(f'Operadores únicos         : {n_ops:,}')
print(f'Clientes únicos (telecom) : {n_clients_t:,}')
print(f'Clientes únicos (clients) : {n_clients_c:,}\n')
# Calculamos o percentual inline para contextualizar o alcance do dataset de chamadas.
print(f'Nota: telecom_df registra {n_clients_t} dos {n_clients_c} clientes cadastrados ({n_clients_t / n_clients_c * 100:.0f}%).')

O `telecom_df` cobre aproximadamente 14 meses de dados, com **1.092 operadores** e **307 clientes ativos** — 42% dos 732 cadastrados no `clients_df`. Os clientes ausentes provavelmente não tiveram atividade no período analisado.

### 2.2 Qualidade dos dados

Verificamos quatro dimensões: **valores ausentes**, **duplicatas**, **tipos de variáveis** e **valores categóricos inesperados**. Por fim, testamos a regra de negócio: `call_duration` ≤ `total_call_duration`.

In [ ]:
# Criamos duas funções para reutilizar a lógica de verificação nos dois datasets
# sem duplicar código — o mesmo padrão de missing_report e duplicate_report se aplica
# a qualquer dataframe, então faz sentido parametrizá-los.

def missing_report(df, name):
    total = len(df)                          # len() retorna o número total de linhas do dataframe
    missing = df.isnull().sum()              # isnull() cria uma máscara booleana (True onde o valor é nulo); .sum() conta os True por coluna
    pct = (missing / total * 100).round(2)   # divide a contagem pelo total de linhas, multiplica por 100 e arredonda para 2 casas decimais
    report = pd.DataFrame({'ausentes': missing, '% do total': pct})  # une as duas séries em um dataframe de fácil leitura
    report = report[report['ausentes'] > 0]  # filtra apenas as colunas com ao menos um nulo — as colunas limpas são ocultadas para não poluir a saída
    print(f'Valores ausentes — {name}:')
    if report.empty:                         # .empty retorna True se o dataframe não tem nenhuma linha (ou seja, não há nulos)
        print('  Nenhum valor ausente.\n')
    else:
        display(report)                      # display() renderiza o dataframe com formatação visual do jupyter


def duplicate_report(df, name):
    # .duplicated() retorna uma série booleana: True para cada linha que é cópia exata de uma linha anterior.
    # .sum() soma os True, retornando o número total de linhas duplicadas.
    n_dup = df.duplicated().sum()
    pct = n_dup / len(df) * 100              # calcula o percentual de duplicatas em relação ao total de linhas
    status = 'ATENÇÃO' if n_dup > 0 else 'OK'  # operador ternário: escolhe o rótulo com base na presença ou ausência de duplicatas
    print(f'Duplicatas — {name}: {n_dup:,} linhas ({pct:.2f}%) [{status}]')


missing_report(telecom_df, 'telecom_df')
missing_report(clients_df, 'clients_df')
duplicate_report(telecom_df, 'telecom_df')
duplicate_report(clients_df, 'clients_df')

In [ ]:
# Verificamos os tipos inferidos automaticamente pelo pandas ao ler os CSVs.
# Isso é essencial antes de qualquer cálculo: somar uma coluna de datas armazenada
# como string, por exemplo, produziria um erro ou resultado silenciosamente incorreto.

print('Tipos de variáveis — telecom_df:')
display(telecom_df.dtypes.rename('dtype'))  # .dtypes retorna uma série com o tipo de cada coluna; .rename() dá título à série para exibição
print('\nTipos de variáveis — clients_df:')  # \n no início da string pula uma linha antes de imprimir
display(clients_df.dtypes.rename('dtype'))

# Construímos manualmente a tabela de conversões necessárias com base na inspeção dos tipos.
# Cada tupla representa: (dataset, coluna, tipo_atual, tipo_esperado, motivo da conversão).
conversions = [
    ('telecom_df', 'date',        'object',  'datetime (utc)',       'contém fuso horário +03:00 — pandas lê como string'),
    ('telecom_df', 'internal',    'object',  'bool',                 'booleano armazenado como string/objeto'),
    ('telecom_df', 'operator_id', 'float64', 'Int64 (nullable int)', 'pandas usa float quando há NaN em coluna de inteiros'),
    ('clients_df', 'date_start',  'object',  'datetime',             'data sem fuso horário, lida como string'),
]
conv_df = pd.DataFrame(            # pd.DataFrame() transforma a lista de tuplas em tabela
    conversions,
    columns=['dataset', 'coluna', 'tipo_atual', 'tipo_esperado', 'observação']
)
print('\nConversões necessárias:')  # \n pula linha antes do título
display(conv_df)

In [ ]:
# Inspecionamos os valores únicos das colunas categóricas para detectar categorias inesperadas,
# erros de digitação ou nulos disfarçados de string (ex.: 'None', 'nan', ' ').
# Usamos um loop para não repetir o mesmo bloco três vezes — um dicionário por coluna é
# adicionado à lista, que depois é convertida em dataframe.

resultados = []  # lista que acumulará um dicionário de informações por coluna analisada

for col in ['direction', 'internal', 'is_missed_call']:
    resultados.append({
        'dataset'       : 'telecom_df',
        'coluna'        : col,
        'valores únicos': str(telecom_df[col].unique().tolist()),  # .unique() retorna array de valores distintos (incluindo NaN); .tolist() converte para lista Python
        'n_únicos'      : telecom_df[col].nunique(dropna=False)    # dropna=False faz NaN contar como uma categoria separada, não ser ignorado
    })

# Adicionamos tariff_plan fora do loop porque pertence a outro dataset.
resultados.append({
    'dataset'       : 'clients_df',
    'coluna'        : 'tariff_plan',
    'valores únicos': str(clients_df['tariff_plan'].unique().tolist()),
    'n_únicos'      : clients_df['tariff_plan'].nunique()
})

display(pd.DataFrame(resultados))  # pd.DataFrame() converte a lista de dicionários em tabela — cada chave vira uma coluna
print("\nObservação: 'internal' possui NaN além dos booleanos esperados — será tratado na limpeza.")

In [ ]:
# Verificamos a regra de negócio: call_duration (duração apenas da conversa)
# nunca deve superar total_call_duration (conversa + tempo de espera na fila).
# Uma violação indicaria inversão dos campos na origem ou erro de coleta.

# A comparação > retorna uma série booleana — True onde a regra é violada.
# .sum() soma os True (cada True vale 1), retornando o número total de violações.
n_inconsistentes = (telecom_df['call_duration'] > telecom_df['total_call_duration']).sum()

print(f'Registros com call_duration > total_call_duration: {n_inconsistentes}')
if n_inconsistentes == 0:   # se não há violações, os dados são consistentes nessa dimensão
    print('Consistência verificada — nenhuma anomalia encontrada.')
else:
    print(f'ATENÇÃO: {n_inconsistentes} registros inconsistentes — investigação necessária.')

**Resumo da qualidade dos dados:**

| Dimensão | Dataset | Achado |
|---|---|---|
| Valores ausentes | `telecom_df` | `operator_id`: 15,16% nulos (8.172 linhas); `internal`: 0,22% nulos (117 linhas) |
| Valores ausentes | `clients_df` | Nenhum valor ausente |
| Duplicatas | `telecom_df` | 4.900 linhas duplicadas (9,09%) |
| Duplicatas | `clients_df` | Nenhuma duplicata |
| Tipagem | `telecom_df` | `date`, `internal` e `operator_id` precisam de conversão |
| Tipagem | `clients_df` | `date_start` precisa de conversão |
| Valores categóricos | `telecom_df` | `internal` contém NaN inesperado |
| Consistência | `telecom_df` | Nenhuma anomalia em `call_duration` vs `total_call_duration` |

Todos os problemas identificados serão resolvidos na próxima etapa de **limpeza e pré-processamento**.

### 2.3 Resumo estatístico

Calculamos as principais estatísticas descritivas das variáveis numéricas e a distribuição das categóricas. Também identificamos os operadores e clientes com maior volume de chamadas perdidas.

In [ ]:
# Calculamos as estatísticas descritivas das variáveis numéricas relevantes para o projeto.
# O resultado de .describe() inclui: count (registros não-nulos), mean (média), std (desvio padrão),
# min, 25º percentil, 50º percentil (mediana), 75º percentil e max.
# Quando o desvio padrão é muito maior que a média, a distribuição é assimétrica (cauda longa à direita),
# o que inviabiliza o teste t (que assume normalidade) e justifica o uso do Mann-Whitney U.

numeric_cols = ['calls_count', 'call_duration', 'total_call_duration']  # selecionamos apenas as colunas numéricas relevantes para os KPIs

display(
    telecom_df[numeric_cols]  # filtra o dataframe mantendo só as colunas de interesse
    .describe()               # calcula as 8 estatísticas resumo para cada coluna
    .round(2)                 # arredonda todos os valores para 2 casas decimais, facilitando a leitura
)

In [ ]:
# Verificamos a distribuição das variáveis categóricas para entender o balanço entre grupos.
# Saber a proporção de chamadas 'in' vs 'out' é fundamental porque alguns KPIs
# se aplicam apenas a um dos grupos (ex.: VCA — volume de chamadas ativas — só para direction='out').

print('Distribuição por direção (direction):')
dir_dist = telecom_df['direction'].value_counts(dropna=False)  # .value_counts() conta ocorrências de cada valor único; dropna=False inclui NaN como categoria
in_pct  = dir_dist.get('in',  0) / len(telecom_df) * 100      # .get('in', 0) retorna 0 se o valor 'in' não existir — evita KeyError
out_pct = dir_dist.get('out', 0) / len(telecom_df) * 100
display(dir_dist.rename('contagem'))                           # .rename() dá nome descritivo à série para a exibição
print(f'  Proporção: in = {in_pct:.1f}% / out = {out_pct:.1f}%\n')  # \n pula linha após o bloco

print('Distribuição por tipo de chamada (internal):')
display(
    telecom_df['internal']
    .value_counts(dropna=False)  # dropna=False é essencial aqui: sem ele, os 117 NaN seriam ignorados e não apareceriam na contagem
    .rename('contagem')
)

print('\nDistribuição por plano tarifário (clients_df):')  # \n no início pula linha antes do título
display(clients_df['tariff_plan'].value_counts().rename('contagem'))

In [ ]:
# Calculamos a frequência de chamadas perdidas e montamos um ranking de operadores/clientes
# com maior volume de perdas. Esse pré-diagnóstico orienta a definição do limiar de ineficiência
# na seção de análise — operadores que aparecem aqui de forma recorrente são candidatos a "ineficientes".

# --- Proporção geral ---
# Usamos normalize=True para obter proporções (0.0 a 1.0) em vez de contagens absolutas,
# o que facilita a comparação independentemente do volume total de registros.
missed_pct = (
    telecom_df['is_missed_call']
    .value_counts(normalize=True)  # normalize=True retorna a fração de cada valor em relação ao total
    * 100                          # multiplicamos por 100 para converter a fração em percentual
)
print('Proporção geral de chamadas perdidas:')
display(
    missed_pct
    .rename(index={True: 'Perdida', False: 'Atendida'})  # substitui True/False por rótulos legíveis para o leitor
    .rename('proporção (%)')                              # dá nome descritivo à série
    .round(2)
)

# --- Top 10 operadores ---
# Agrupamos por operator_id e somamos calls_count (não contamos linhas),
# porque cada linha representa múltiplas chamadas agregadas — somar linhas subestimaria o volume real.
print('\nTop 10 operadores por volume de chamadas perdidas:')
top_missed_ops = (
    telecom_df[telecom_df['is_missed_call']]   # filtra apenas registros onde a chamada foi perdida (is_missed_call == True)
    .groupby('operator_id')['calls_count']     # agrupa por operador e seleciona a coluna de quantidade de chamadas
    .sum()                                     # soma o total de chamadas perdidas por operador
    .sort_values(ascending=False)              # ordena do maior para o menor volume
    .head(10)                                  # mantém apenas os 10 operadores com maior volume
    .rename('chamadas_perdidas')
)
display(top_missed_ops.to_frame())  # .to_frame() converte a série em dataframe de uma coluna para melhor visualização

# --- Top 5 clientes ---
# Mesma lógica aplicada por cliente — útil para saber se o problema está concentrado
# em poucos clientes ou distribuído de forma uniforme.
print('\nTop 5 clientes por volume de chamadas perdidas:')
top_missed_clients = (
    telecom_df[telecom_df['is_missed_call']]
    .groupby('user_id')['calls_count']
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .rename('chamadas_perdidas')
)
display(top_missed_clients.to_frame())

**Principais achados estatísticos:**

- **`calls_count`**: mediana de 4 chamadas, média de 16,5 — fortemente influenciada por outliers (máximo de 4.817). Distribuição altamente assimétrica.
- **`call_duration` e `total_call_duration`**: mesmo padrão — medianas baixas (38 s e 210 s) contra médias muito superiores (867 s e 1.157 s), indicando chamadas longas que puxam a média.
- A alta variabilidade em todas as métricas confirma que os dados não seguem distribuição normal, o que reforça a escolha do teste **Mann-Whitney U** na validação das hipóteses.

### 2.4 Variáveis mais relevantes

Com base no diagnóstico, as variáveis prioritárias para a análise são:

| Variável | Papel na análise |
|---|---|
| `operator_id` | Variável-chave de agregação — todos os KPIs são calculados por operador |
| `is_missed_call` + `calls_count` | Base para calcular a Taxa de Chamadas Perdidas (TCP) |
| `total_call_duration` − `call_duration` | Tempo de espera por chamada — base do TME (variável derivada) |
| `direction` | Distingue operadores de entrada (`in`) e saída (`out`) |
| `internal` | Separa chamadas internas das externas na análise de perdas |
| `tariff_plan` | Contextualiza o cliente — pode influenciar o perfil de chamadas |

> **Próximo passo:** limpeza e pré-processamento — conversão de tipos, tratamento de nulos e remoção de duplicatas.